# AeroPredict Argentina — EDA y visualizaciones (módulo de DATOS)

Análisis exploratorio sobre los entregables del pipeline (`data/processed/`).
Cada gráfico se guarda también en `reports/figuras/` para usar en la presentación.

Requisito: haber corrido `python -m src.pipeline` (genera los CSV).


In [ ]:
%matplotlib inline
import sys; sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from src import config

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
M = mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M')

FIG_DIR = config.REPORTS_DIR / 'figuras'
FIG_DIR.mkdir(parents=True, exist_ok=True)
def guardar(fig, nombre): fig.savefig(FIG_DIR / nombre, dpi=120, bbox_inches='tight')

# Entregables
m = pd.read_csv(config.SALIDA_MENSUAL_RUTA)
aero = pd.read_csv(config.SALIDA_AEROPUERTOS)
m['fecha'] = pd.to_datetime(m['anio'].astype(str) + '-' + m['mes'].astype(str).str.zfill(2) + '-01')
print('base_mensual_ruta:', m.shape, '| aeropuertos:', aero.shape)
m.head()

## 1. Dashboard general (Módulo 1)

In [ ]:
total_pax = int(m.pasajeros.sum())
total_asi = int(m.asientos.sum())
total_vue = int(m.vuelos.sum())
print(f"Pasajeros totales : {total_pax:,}")
print(f"Asientos totales  : {total_asi:,}")
print(f"Vuelos totales    : {total_vue:,}")
print(f"Factor ocupación  : {total_pax/total_asi:.1%}")
print(f"Rutas únicas      : {m.ruta.nunique():,}")
print(f"Aeropuertos (5.4) : {len(aero)}")
print(f"Período           : {m.anio.min()}–{m.anio.max()}")

In [ ]:
fig, ax = plt.subplots()
m.groupby(['fecha', 'clasificacion_vuelo']).pasajeros.sum().unstack().plot(ax=ax)
m.groupby('fecha').pasajeros.sum().plot(ax=ax, color='black', lw=2, label='Total')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'),
           color='red', alpha=0.1, label='Pandemia')
ax.set_title('Evolución mensual de pasajeros')
ax.set_ylabel('Pasajeros'); ax.set_xlabel('')
ax.yaxis.set_major_formatter(M); ax.legend()
guardar(fig, '01_evolucion_mensual.png'); plt.show()

## 2. Análisis de rutas (Módulo 2)

In [ ]:
top = m.groupby('ruta').pasajeros.sum().sort_values().tail(15)
fig, ax = plt.subplots(figsize=(10, 7))
top.plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 15 rutas por pasajeros (acumulado 2017–2026)')
ax.set_xlabel('Pasajeros'); ax.set_ylabel('')
ax.xaxis.set_major_formatter(M)
guardar(fig, '02_top_rutas.png'); plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(m.factor_ocupacion, bins=40, color='teal', alpha=0.8)
ax.axvline(config.UMBRAL_BAJA_OCUPACION, color='red', ls='--',
           label=f'Baja < {config.UMBRAL_BAJA_OCUPACION}')
ax.axvline(config.UMBRAL_OCUPACION_ELEVADA, color='green', ls='--',
           label=f'Elevada ≥ {config.UMBRAL_OCUPACION_ELEVADA}')
ax.set_title('Distribución del factor de ocupación (ruta-mes)')
ax.set_xlabel('Factor de ocupación'); ax.set_ylabel('Cantidad de ruta-mes')
ax.legend()
guardar(fig, '03_factor_ocupacion.png'); plt.show()

In [ ]:
g = m.groupby('clasificacion_vuelo').agg(
    pasajeros=('pasajeros', 'sum'), asientos=('asientos', 'sum'), rutas=('ruta', 'nunique'))
g['factor'] = g.pasajeros / g.asientos
print(g.round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
g.pasajeros.plot.bar(ax=axes[0], color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('Pasajeros totales'); axes[0].set_xlabel(''); axes[0].yaxis.set_major_formatter(M)
(g.factor * 100).plot.bar(ax=axes[1], color=['#1f77b4', '#ff7f0e'])
axes[1].set_title('Factor de ocupación (%)'); axes[1].set_xlabel('')
for ax in axes: ax.tick_params(axis='x', rotation=0)
guardar(fig, '04_cabotaje_vs_internacional.png'); plt.show()

In [ ]:
est = m.groupby(['anio', 'mes']).pasajeros.sum().groupby('mes').mean()
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
fig, ax = plt.subplots()
ax.bar(range(1, 13), est.reindex(range(1, 13)).values, color='darkorange')
ax.set_xticks(range(1, 13)); ax.set_xticklabels(meses)
ax.set_title('Estacionalidad: pasajeros promedio por mes (feature clave del modelo)')
ax.set_xlabel(''); ax.set_ylabel('Pasajeros (promedio mensual)')
ax.yaxis.set_major_formatter(M)
guardar(fig, '05_estacionalidad.png'); plt.show()

## 3. Alertas — baja ocupación (Módulo 4)

In [ ]:
baja = m[m.factor_ocupacion < config.UMBRAL_BAJA_OCUPACION]
print(f"Ruta-mes con baja ocupación (<{config.UMBRAL_BAJA_OCUPACION}): "
      f"{len(baja):,} de {len(m):,} ({len(baja)/len(m):.1%})")
print('\nRutas con baja ocupación más recurrente (cantidad de meses):')
print(baja.groupby('ruta').size().sort_values(ascending=False).head(10).to_string())

## 4. Mapa de aeropuertos (Módulo 5, opcional)

In [ ]:
pax_loc = pd.concat([
    m.groupby('origen_localidad').pasajeros.sum(),
    m.groupby('destino_localidad').pasajeros.sum(),
], axis=1).sum(axis=1)
a = aero.copy()
a['pasajeros'] = a.localidad.map(pax_loc).fillna(0)
arg = a[a.latitud.between(-56, -20) & a.longitud.between(-74, -52)]
fig, ax = plt.subplots(figsize=(6, 9))
sc = ax.scatter(arg.longitud, arg.latitud,
                s=(arg.pasajeros / arg.pasajeros.max() * 600) + 15,
                c=arg.pasajeros, cmap='viridis', alpha=0.75, edgecolor='k', linewidth=0.3)
for _, r in arg.sort_values('pasajeros').tail(6).iterrows():
    ax.annotate(r.localidad, (r.longitud, r.latitud), fontsize=8,
                xytext=(4, 4), textcoords='offset points')
ax.set_title('Aeropuertos argentinos por tráfico de pasajeros')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
fig.colorbar(sc, ax=ax, label='Pasajeros', shrink=0.6)
guardar(fig, '06_mapa_aeropuertos.png'); plt.show()

## 5. Notas para el modelo predictivo (equipo de IA)

- **Quiebre 2020–2021 (pandemia):** caída brusca visible en la evolución mensual. Excluir del entrenamiento o tratar explícitamente.
- **Estacionalidad anual marcada:** `mes` es feature clave (verano = picos). Codificar cíclico o categórico.
- **Features sugeridos:** lag de pasajeros del mismo mes del año anterior, promedio histórico de la ruta.
- Reporte de calidad completo en `reports/calidad_datos.md`.